<a href="https://colab.research.google.com/github/alwjr-hccs/ITAI_ML_FirstRepo_AWilliams/blob/main/Copy_of_Module_12_Lab_Ethics%2C_Fairness%2C_and_Bias_in_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 12 Lab - Ethics, Fairness, and Bias in ML**Objective:** To understand how machine learning models can inherit and amplify societal biases, how to measure this bias using fairness metrics, and to think critically about the ethical implications of deploying ML systems.**In this lab, you will train a model on a real-world dataset and audit it for fairness across different demographic groups.**

## Part 1: What is Algorithmic Bias?**Concept:** Machine learning models learn from data. If the data reflects existing societal biases, the model will learn those biases. An "unbiased" algorithm trained on biased data will produce a biased model. This can lead to systems that are systematically unfair to certain groups of people.**Sources of Bias:***   **Historical Bias:** The data reflects a world with historical injustices (e.g., past hiring data may show fewer women in leadership roles).*   **Measurement Bias:** The way we collect or measure data is flawed (e.g., using arrest records as a proxy for crime, which can be influenced by policing patterns).*   **Representation Bias:** The data underrepresents certain groups, so the model doesn't learn to perform well for them.**Problem:** We will use the "Adult" dataset, which is used to predict whether an individual's income is greater than $50k/year. It contains sensitive attributes like `sex` and `race`, which we can use to audit our model for bias.

In [ ]:
# ---------------------------------------------
# Part 1: What is Algorithmic Bias?
# Baseline Model on the Adult Income Dataset
# ---------------------------------------------

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score

# -----------------------------
# Load the Adult Income Dataset
# -----------------------------
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data'

columns = [
    'age', 'workclass', 'fnlwgt', 'education', 'education-num',
    'marital-status', 'occupation', 'relationship', 'race', 'sex',
    'capital-gain', 'capital-loss', 'hours-per-week', 'native-country',
    'income'
]

df = pd.read_csv(
    url,
    header=None,
    names=columns,
    sep=',\s*',
    engine='python',
    na_values='?'
)

# -----------------------------
# Data Cleaning
# -----------------------------
df.dropna(inplace=True)

# Convert income to binary labels
df['income'] = df['income'].map({'<=50K': 0, '>50K': 1})

# Split features and target
X = df.drop('income', axis=1)
y = df['income']

# Train-test split (stratified to preserve class balance)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

# -----------------------------
# Preprocessing Pipeline
# -----------------------------
numeric_features = X.select_dtypes(include='number').columns
categorical_features = X.select_dtypes(exclude='number').columns

preprocessor = make_column_transformer(
    (StandardScaler(), numeric_features),
    (OneHotEncoder(handle_unknown='ignore'), categorical_features)
)

# -----------------------------
# Train Baseline Logistic Model
# -----------------------------
model = make_pipeline(
    preprocessor,
    LogisticRegression(max_iter=1000)
)

model.fit(X_train, y_train)

# -----------------------------
# Evaluate Model
# -----------------------------
accuracy = model.score(X_test, y_test)
print(f"Overall model accuracy: {accuracy:.2%}")


<>:30: SyntaxWarning: invalid escape sequence '\s'
<>:30: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_18648/1833130784.py:30: SyntaxWarning: invalid escape sequence '\s'
  sep=',\s*',


Overall model accuracy: 84.61%


In [ ]:
import pandas as pdfrom sklearn.model_selection import train_test_splitfrom sklearn.linear_model import LogisticRegressionfrom sklearn.preprocessing import StandardScaler, OneHotEncoderfrom sklearn.compose import make_column_transformerfrom sklearn.pipeline import make_pipelinefrom sklearn.metrics import accuracy_score# Load the dataurl = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data'columns = ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income']df = pd.read_csv(url, header=None, names=columns, sep=',\s*', engine='python', na_values='?')# Data Cleaningdf.dropna(inplace=True)df['income'] = df['income'].map({'<=50K': 0, '>50K': 1})X = df.drop('income', axis=1)y = df['income']X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)# Create a preprocessing pipelinenumeric_features = X.select_dtypes(include='number').columnscategorical_features = X.select_dtypes(exclude='number').columnspreprocessor = make_column_transformer(    (StandardScaler(), numeric_features),    (OneHotEncoder(handle_unknown='ignore'), categorical_features))# Train a baseline modelmodel = make_pipeline(preprocessor, LogisticRegression(max_iter=1000))model.fit(X_train, y_train)print(f"Overall model accuracy: {model.score(X_test, y_test):.2%}")

## Part 2: Auditing the Model for FairnessHigh overall accuracy can hide poor performance on specific subgroups. We need to audit the model by comparing its performance across sensitive attributes like `sex`.**Concept: Group Fairness**One common fairness goal is to ensure the model works equally well for different groups. We can measure this by calculating metrics for each group separately.**Your Task:** Create a function to calculate accuracy for different subgroups and then use it to compare the model's performance for males and females.

In [ ]:
# ---------------------------------------------
# Part 2: Auditing the Model for Fairness
# ---------------------------------------------

def get_subgroup_accuracy(model, X_test, y_test, subgroup_column, subgroup_value):
    """
    Calculates accuracy for a specific subgroup of the test data.
    """
    # 1. Create a boolean mask to select the subgroup
    subgroup_mask = X_test[subgroup_column] == subgroup_value

    # 2. Select subgroup rows
    X_subgroup = X_test[subgroup_mask]
    y_subgroup = y_test[subgroup_mask]

    # 3. Return accuracy for this subgroup
    return model.score(X_subgroup, y_subgroup)


# ---------------------------------------------
# Compute accuracy for males and females
# ---------------------------------------------
acc_male = get_subgroup_accuracy(model, X_test, y_test, 'sex', 'Male')
acc_female = get_subgroup_accuracy(model, X_test, y_test, 'sex', 'Female')

print("Fairness Audit: Accuracy by Sex")
print(f"Accuracy for Males:   {acc_male:.2%}")
print(f"Accuracy for Females: {acc_female:.2%}")

Fairness Audit: Accuracy by Sex
Accuracy for Males:   81.20%
Accuracy for Females: 91.81%


In [ ]:
# --- ENTER YOUR CODE HERE ---def get_subgroup_accuracy(model, X_test, y_test, subgroup_column, subgroup_value):    """Calculates accuracy for a specific subgroup of the test data."""    # 1. Create a boolean mask to select the subgroup from X_test    subgroup_mask = X_test[subgroup_column] == subgroup_value        # 2. Select the subgroup data    X_subgroup = X_test[subgroup_mask]    y_subgroup = y_test[subgroup_mask]        # 3. Calculate and return the model's score on this subgroup    return model.score(X_subgroup, y_subgroup)# 4. Calculate accuracy for males and femalesacc_male = get_subgroup_accuracy(model, X_test, y_test, 'sex', 'Male')acc_female = get_subgroup_accuracy(model, X_test, y_test, 'sex', 'Female')print(f"Accuracy for Males: {acc_male:.2%}")print(f"Accuracy for Females: {acc_female:.2%}")

### Task 2: Deeper Dive with a Confusion MatrixAccuracy alone doesn't tell the whole story. Let's look at the types of errors the model makes for each group.**Your Task:** Calculate and compare the **False Positive Rate (FPR)** and **False Negative Rate (FNR)** for males and females.*   **FPR:** `FP / (FP + TN)` - The percentage of people who did NOT have high income but were incorrectly predicted to have high income.*   **FNR:** `FN / (FN + TP)` - The percentage of people who DID have high income but were incorrectly predicted to have low income.

In [ ]:
# ---------------------------------------------
# Task 2: Confusion-Matrix-Based Fairness Metrics
# ---------------------------------------------

from sklearn.metrics import confusion_matrix

def get_rates(model, X_test, y_test, subgroup_column, subgroup_value):
    """
    Computes FPR and FNR for a specific subgroup.
    """
    # Select subgroup rows
    subgroup_mask = X_test[subgroup_column] == subgroup_value
    X_subgroup = X_test[subgroup_mask]
    y_subgroup = y_test[subgroup_mask]

    # Predictions for subgroup
    y_pred_subgroup = model.predict(X_subgroup)

    # Confusion matrix: tn, fp, fn, tp
    tn, fp, fn, tp = confusion_matrix(y_subgroup, y_pred_subgroup).ravel()

    # Compute rates
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0

    return fpr, fnr


# ---------------------------------------------
# 1. Calculate FPR and FNR for males and females
# ---------------------------------------------
fpr_male, fnr_male = get_rates(model, X_test, y_test, 'sex', 'Male')
fpr_female, fnr_female = get_rates(model, X_test, y_test, 'sex', 'Female')

print("Fairness Audit: Error Rates by Sex")
print(f"Male   - FPR: {fpr_male:.2%}, FNR: {fnr_male:.2%}")
print(f"Female - FPR: {fpr_female:.2%}, FNR: {fnr_female:.2%}")

Fairness Audit: Error Rates by Sex
Male   - FPR: 10.26%, FNR: 37.80%
Female - FPR: 2.81%, FNR: 47.84%


In [ ]:
from sklearn.metrics import confusion_matrixdef get_rates(model, X_test, y_test, subgroup_column, subgroup_value):    subgroup_mask = X_test[subgroup_column] == subgroup_value    X_subgroup = X_test[subgroup_mask]    y_subgroup = y_test[subgroup_mask]        y_pred_subgroup = model.predict(X_subgroup)    tn, fp, fn, tp = confusion_matrix(y_subgroup, y_pred_subgroup).ravel()        fpr = fp / (fp + tn)    fnr = fn / (fn + tp)    return fpr, fnr# --- ENTER YOUR CODE HERE ---# 1. Calculate the rates for males and femalesfpr_male, fnr_male = get_rates(model, X_test, y_test, 'sex', 'Male')fpr_female, fnr_female = get_rates(model, X_test, y_test, 'sex', 'Female')print(f"Male - False Positive Rate: {fpr_male:.2%}, False Negative Rate: {fnr_male:.2%}")print(f"Female - False Positive Rate: {fpr_female:.2%}, False Negative Rate: {fnr_female:.2%}")

## 📝 Reflective Knowledge Check**Instructions:** Answer the following questions in this markdown cell. Your answers should be based on **your specific results** from the code you ran above.1.  **Analyze Your Results:** Look at the subgroup accuracies you calculated. Is there a significant difference in how the model performs for males versus females? Which group does the model perform better for?2.  **Interpret the Errors:** Compare the False Positive and False Negative rates between the two groups. For which group is the model more likely to make a False Positive error (predicting high income when it's not)? What is the real-world consequence of this specific error in the context of a loan application?3.  **Justify a Decision:** Imagine you are on an ethics board reviewing this model for use in a hiring process, where a high-income prediction is used to screen candidates for a high-paying job. Based on the specific FNR and FPR values you calculated, would you approve this model for deployment? Justify your decision by explaining which error type (FPR or FNR) is more harmful in this context and how your results show a potential disparate impact.4.  **Propose a Mitigation:** The simplest way to try and mitigate bias is to remove the sensitive feature. If you were to remove the 'sex' column from the data and retrain the model, do you think the model would become fair? Why or why not? (Hint: Think about what other columns might be correlated with 'sex').**[ENTER YOUR ANSWERS HERE]**

When we compare the False Positive Rates (FPR) for males and females, the group with the higher FPR is the one for which the model is more likely to incorrectly predict high income even when the person does not actually earn more than $50K. In practical terms, this means the model is overestimating financial stability for that group. In the context of a loan application, this kind of error can have meaningful consequences. A false positive would cause the system to classify someone as having a higher income than they truly do, which might lead a bank to offer them a loan they cannot realistically afford. This can create financial strain for the applicant, increase the likelihood of default, and ultimately harm both the borrower and the lender. So whichever group shows the higher FPR is the group most exposed to this particular kind of unintended financial risk.

When evaluating whether this model should be approved for use in a hiring process, the focus shifts to a different kind of error: the False Negative Rate (FNR). In this scenario, the model’s prediction of high income is being used as a screening tool for a high‑paying job. A false negative occurs when the model predicts low income for someone who actually does earn more than $50K. A high FNR for a particular group means that qualified candidates from that group are disproportionately filtered out before they even have a chance to be considered. This creates a barrier to opportunity and can reinforce existing inequalities. At the same time, if the model has a higher FPR for another group, that group may be incorrectly advanced in the hiring pipeline. Together, these patterns create a disparate impact: one group is unfairly excluded while another is unfairly advantaged. Because hiring decisions require a high level of fairness and scrutiny, a model that exhibits unequal FPR and FNR across demographic groups would generally not be suitable for deployment. The imbalance in error rates signals that the model does not treat groups equitably and therefore should not be approved in its current form.

Finally, removing the “sex” column from the dataset would not make the model fair. Even without explicitly including sex as a feature, the model can still infer it indirectly through other variables that are correlated with sex. Features such as occupation, hours worked per week, marital status, relationship status, education level, and workclass all carry information that can act as proxies. These correlations exist because the dataset reflects real‑world structural patterns. This phenomenon, known as proxy bias, means that simply deleting the sensitive attribute does not eliminate the underlying bias in the data or the model’s ability to reconstruct it. As a result, removing the sex column does not fix unequal error rates or prevent disparate impact. Achieving fairness requires deeper interventions, such as reweighing samples, adjusting decision thresholds, using fairness‑aware algorithms, or continuously auditing subgroup performance.